# Step 09 — each site splits its own patients into discovery and validation

**Data type: RNA_array** (GSE65391). **Reads:** `step06_site_{A,B,C}.rds` (federated ComBat),
`cohorts/split_assignment.csv`. **Writes:** `step09_site_{A,B,C}.rds`.

Endotypes are found on 70% of the patients (**discovery**) and checked on the other 30%
(**validation**). The split is by **patient**, not by sample, so the repeated visits of one child
never sit on both sides. Otherwise a child's later visit would be "validated" against a cluster built
from their own earlier visit, which is circular.

Each site draws its own 30%. Nothing is exchanged. Healthy children are not split: they are the
reference for the interferon score, not patients to cluster.

**Frozen:** written once to `cohorts/split_assignment.csv` and read back on every run.

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("09")
SPLIT_FILE <- coh("split_assignment.csv")
frozen <- if (file.exists(SPLIT_FILE)) read.csv(SPLIT_FILE, stringsAsFactors = FALSE) else NULL
new_rows <- list()
for (i in seq_along(SITES)) {
  s <- SITES[i]
  d <- readRDS(site_file("06", s))
  pts <- sort(unique(d$meta$subject[d$meta$disease == "SLE"]))
  if (!is.null(frozen)) {
    sp <- frozen[frozen$site == s, ]
    stopifnot(setequal(sp$subject, pts))
  } else {
    set.seed(SEED + 3L + i)
    sp <- data.frame(subject = pts, site = s, split = "discovery")
    sp$split[sample(length(pts), round(0.3 * length(pts)))] <- "validation"
    new_rows[[s]] <- sp
  }
  d$meta$split <- ifelse(d$meta$disease == "SLE", setNames(sp$split, sp$subject)[d$meta$subject], "healthy")
  saveRDS(d, site_file("09", s))
  send(table(d$meta$split), s, "sample counts per split", ncol(d$E))
  print(c(site = s, table(patients = sp$split)))
}
if (is.null(frozen)) write.csv(do.call(rbind, new_rows), SPLIT_FILE, row.names = FALSE)

      site  discovery validation 
       "A"       "37"       "16" 
      site  discovery validation 
       "B"       "37"       "16" 
      site  discovery validation 
       "C"       "36"       "16" 


In [2]:
# samples, not patients: every visit follows its patient
sapply(SITES, function(s) table(readRDS(site_file("09", s))$meta$split))

,A,B,C
discovery,238,190,247
healthy,16,16,16
validation,84,74,91


## Findings

Each site holds about 37 discovery and 16 validation patients. Every visit of a patient stays on that
patient's side of the split.